#### This is an explanation of the operations the notebook does
---
The cell bellow makes the necessary imports for the libraries that are going to be used, and establishes a connection with the data lake storage system -minio- by calling the appropriate function that already exists in the configuration file of the framework

In [ ]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config

client = config.create_minio_client()

### Step 0: Data Ingestion from MinIO

In the cell below, the MinIO object storage is accessed through the client created in the previous step. A specific raw JSON object representing a Google Cloud service is downloaded and stored in the corresponding variable. 

Afterwards, using pandas, this raw JSON is transformed into an initial DataFrame. At this starting point, the DataFrame contains only two top-level key-value pairs:
1. **`skus`**: A nested list containing all the individual services/products (`[list of services]`).
2. **`nextPageToken`**: A string token used for API pagination (`"String"`).

In [ ]:
"""
Kubernetes_Engine: 1 page, 1.4 Mib, 1731rows, produces 26 columns
Compute_Engine: 7 pages, 4.1Mib and 1.1Mib, 5000 rows/page, produces 29 columns
Cloud_Storage: 1 page, 1000Kib, 1220 services, 3000 with duplicates, produces 29 columns
Cloud_SQL : 4 pages, produces 29 columns
Networking: 1 page, 1600 services, 2800 non unique, produces 29 columns

"""
object_name = "Cloud_SQL/page_4.json"

try:
    response = client.get_object(config.PROVIDERS.get("google").get("bucket"), object_name=object_name)

    df = pd.read_json(response)

    response.close()
    response.release_conn()

    print ("Success. Page loaded and converted into dataframe")
    print ("Array size: Rows = ",df.shape[0], " and Columns = ", df.shape[1])

except Exception as e:
    print ("Error: ",e)


### Step 1: Loading & Initial Flattening

In the cell below, the first key-value pair is flattened and the list of services is opened, with each service occupying a row. Also, any first-level inner dictionaries (dicts) that a service may contain are also opened.

For example, a nested structure like this:
`category: {key1: value1, key2: value2}`

Opens up and flattens into distinct columns:
`category.key1`, `category.key2` with their respective values mapped across the rows.

In [ ]:
#With the commnand bellow, we open the first level key : value pairs in columns, and the first level inner dicts also open, 
#in the form key.value (ex: category.serviceDisplayName, <- This was an inner dict category :{key:value, key:value})

df_flat = pd.json_normalize(df['skus'])
df_flat.head(3)
# df_flat[['skuId', 'category.serviceDisplayName', 'category.resourceFamily', 'category.usageType', 'category.resourceGroup']].head()

### Step 2: Exploding Geographic Regions

In the cell below, we perform an `.explode()` operation on the `geoTaxonomy.regions` column, which originally contains a nested list of locations. 

By exploding this list, each region occupies its own row. This means that for services available in multiple locations, duplicated rows are created for all other attributes, with the only difference being the specific region value in each row.

In [ ]:
#Here geoTaxonomy.regions looks like this: ["value"]
df_flat[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()

#We apply the explode in the geoTaxonomy.regions column and store the res in a new dataframe
df_flat2 = df_flat.explode('geoTaxonomy.regions')

#The result will be: geoTaxonomy wont be a list anymore, and all the list elements will be in a single line
# df_flat2[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()
df_flat2.head(3)


### Step 3: Exploding Service Regions

In the cell below, we perform the exact same `.explode()` operation on the service regions column. 

Just like with the geotaxonomy regions, this operation unpacks the nested list of service regions into individual rows, resulting in duplicated rows.

In [ ]:
#We apply the same proccedure as the cell above. This time we open the list serviceRegions
df_flat2[['skuId', 'serviceRegions']].head()

df_flat3 = df_flat2.explode('serviceRegions')

df_flat3[['skuId', 'serviceRegions', 'geoTaxonomy.regions']].head()

df_flat3.head()

### Step 4: Exploding Pricing information

Just like the 2 cells above an `.explode()` operation is made to unpack the list of `pricingInfo`, and isolate each of the elements in a single row. For the record, the value within the `pricingInfo` list is a dictionary, which contains the detailed pricing structures and rates for each service

In [ ]:
#The only column not fully opened yet is the pricingInfo: structure -> [{key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}}]
df_flat3[['skuId', 'pricingInfo']].head()

#This first explode removes the list: we have now -> {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} 
df_flat4 = df_flat3.explode('pricingInfo')
df_flat4[['skuId', 'pricingInfo']].head()
df_flat4.head()

### Step 5: Flattening the pricingInfo Dictionary

At this stage of the pipeline, the pricingInfo column contains a dict with the following structure:
`{key1: val1, key2: val2, key3: val3, pricingExpression: {key: val, tieredRates: [{}]}}`

To extract these nested properties into standalone columns, we use the `pd.json_normalize()` again.

**What this operation does:**
1. **Unpacks the Dictionary:** It takes the keys of the `pricingInfo` dictionary (such as `effectiveTime`, `summary`, `currencyConversionRate` and `pricingExpression`) and turns them into separate, clean columns.
2. **Handles Sub-Nested Structures:** If a key contains further nested dictionaries (like `pricingExpression`), it automatically flattens them using dot notation (e.g., `pricingExpression.pricingUnits`, `pricingExpression.baseUnit`).
3. **Preserves Sub-Lists:** Deeply nested lists (like `pricingExpression.tieredRates` which contains the actual price tiers) are kept intact inside their cells, preparing them for the final cleaning steps.

In [ ]:
#We are here now: {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} -> we can open the dict with the normalize
#A new dataframe will be created with the pricing info and then concatenated with the original dataframe

pricing1 = pd.json_normalize(df_flat4['pricingInfo'])
pricing1.head()

### Step 6: Aligning Indexes and Merging Data

Since `normalize()` creates a new DataFrame (`pricing1` in our case) we have  to merge it back with our main DataFrame to reconstruct the complete dataset. During this concatenation it is imperative to align the indexes of each line, and drop the pricingInfo column from the first dataset, since it's data will now be in standalone cols


In [ ]:
#I now have to concatenate the 2 dataframes beeing carefull though with the indexes
pricing1.index = df_flat4.index

df_flat5 = pd.concat([df_flat4.drop(columns=['pricingInfo']), pricing1], axis=1)
df_flat5.head()
# df_flat5['pricingExpression.tieredRates']

### Step 7: Exploding Pricing Tiers

In the cell below, we perform the final `.explode()` operation on the `pricingExpression.tieredRates` column. 
Since a single SKU can have multiple pricing tiers, this operation unpacks the nested list of tiers into individual rows, ensuring every distinct price rate is isolated for analysis.
The value of the list was dictionary/ries so the next operation that is requires is, flattening that dictionary and then concatenation. Those operations take place in the 3 cells bellow -the detailed procedure is not explained as is simillar with above-

In [ ]:
df_flat6 = df_flat5.explode('pricingExpression.tieredRates')

df_flat6[['skuId', 'pricingExpression.tieredRates']].head()

In [ ]:
pricing2 = pd.json_normalize(df_flat6['pricingExpression.tieredRates'])
pricing2.head()

In [ ]:
pricing2.index = df_flat6.index

df_final_flat = pd.concat([df_flat6.drop(columns=['pricingExpression.tieredRates']), pricing2], axis=1)
pd.set_option('display.max_columns', None)
df_final_flat.head()

### Step 8: Calculating the Final Price (Handling Units & Nanos)

According to Google Cloud's official documentation, SKU prices are not represented as standard floats. Instead, they are split into two separate components to prevent floating-point precision errors:
* `units`: The whole number part of the price.
* `nanos`: The fractional part of the price, represented as billionths of a cent/dollar ($10^{-9}$).

For example, a price of **$1.75** is stored as `units = 1` and `nanos = 750,000,000`.

**In the cell below, we:**: reconstruct the price by converting both columns to floats, divide `unitPrice.nanos` by $1,000,000,000$, and add them together to create the clean `finalPrice` column.


In [ ]:
#The cost of the SKU is units + nanos. For example, a cost of $1.75 is represented as units=1 and nanos=750,000,000. (From google documentation)
#Actions need to be made to create a new column that will contain the final price
df_final_flat['finalPrice'] = df_final_flat['unitPrice.units'].astype(float) + (df_final_flat['unitPrice.nanos'].astype(float) / 1000000000)

df_final_flat[['skuId', 'unitPrice.units', 'unitPrice.nanos', 'finalPrice']]

In [ ]:
#Just for debug to search if any line has "1" as unit price
filtered_df = df_final_flat[df_final_flat['unitPrice.units'].astype(float) == 1.0]

filtered_df[['skuId','unitPrice.units', 'unitPrice.nanos', 'finalPrice']].head()

# df_final_flat.head()

In [ ]:
print(df_final_flat['unitPrice.currencyCode'].value_counts())
print ()
print(df_final_flat.shape[0])
print()
print(df_final_flat['skuId'].nunique())

In [ ]:
#Just fot debug: filtering lines where the currencny is not USD
eur_rows_df = df_final_flat[df_final_flat['unitPrice.currencyCode'] != 'USD']

print(df_final_flat['unitPrice.currencyCode'].value_counts(dropna=False))

eur_rows_df.head()

### Step 9: Cleaning, Currency normalization, Duplicate removal

**Operations Performed:**
1. We remove any rows where `unitPrice.currencyCode` is absent, filtering out unpriced or incomplete SKU records.

2. We convert all prices into (USD) by dividing the `finalPrice` by the provided `currencyConversionRate`. Easier for future analytics

3. For data profiling purposes we create a separate subset that keeps only the first occurrence of each unique `skuId`. This ensures accurate statistical metrics.


In [ ]:
#Throw duplicates, throw records with Nan in currencycode, new column with usd final price

df_clean_no_nan = df_final_flat.dropna(subset=['unitPrice.currencyCode']).copy() #If the value is nan in this column drop (If the product has no currency registered its problematic)

#Currency normalization
rate = df_clean_no_nan['currencyConversionRate'].astype(float)

#This is our clean array so far
df_clean_no_nan['final_price_usd'] = df_clean_no_nan['finalPrice'].astype(float) / rate

#Only for profiling tool drop all duplicates
df_for_profiling = df_clean_no_nan.drop_duplicates(subset=['skuId'])

print(f"The size of the array is {df_clean_no_nan.shape[0]} rows, and {df_clean_no_nan.shape[1]}, columns")
print (f"The size of the array used for profiling is {df_for_profiling.shape[0]} rows and {df_for_profiling.shape[1]} columns")
df_clean_no_nan.head()
# print(df_for_profiling.shape[0])

In [ ]:
# from ydata_profiling import ProfileReport

# profile = ProfileReport(df_for_profiling, title="Google Cloud Compute Engine - Unique SKUs Report", explorative=True)

# #Stores in file under the same directory
# profile.to_file("google_billing_unique_analysis.html")

# print("Report created")

**Debug purposes**

In [ ]:
#To check witch of the aggreagation columns actually have a value. Most of them dont
df_clean_no_nan[df_clean_no_nan['aggregationInfo.aggregationCount'].notna()]